# LAB 03 — Construye tu búsqueda semántica

**Objetivo:** visualizar **Query → Embedding → Similarity → Top-K**.

**Tiempo sugerido:** 45 minutos · **Requiere proveedor configurado** (OpenAI u Ollama) a partir de la sección 3.

Pasos: 1) crear embeddings · 2) guardarlos · 3) convertir la pregunta en embedding ·
4) comparar contra los documentos · 5) ordenar · 6) obtener Top-K.

In [ ]:
# Verificación del entorno: este notebook debe correr con el kernel .venv del repositorio
import sys, importlib.util
_faltan = [p for p in ["numpy", "pandas", "sklearn", "tiktoken", "openai", "dotenv", "matplotlib"]
           if importlib.util.find_spec(p) is None]
if _faltan:
    raise ModuleNotFoundError(
        f"Faltan paquetes: {', '.join(_faltan)}\n"
        f"Este kernel usa: {sys.executable}\n"
        "Solución: en la terminal, dentro del repositorio, ejecuta `uv sync`; luego en VS Code elige "
        "Select Kernel → Python Environments → .venv (la del repositorio) y reinicia el kernel."
    )
print("✅ Entorno listo:", sys.executable)

In [ ]:
# Configuración común: localiza el repositorio, carga .env y el documento de trabajo
import os
import json
from pathlib import Path
from dotenv import load_dotenv

def encontrar_raiz():
    p = Path.cwd().resolve()
    for c in [p, *p.parents]:
        if (c / "data").is_dir() and (c / "pyproject.toml").exists():
            return c
    raise FileNotFoundError("No encontré la carpeta data/. Abre el notebook desde el repositorio del curso.")

ROOT = encontrar_raiz()
load_dotenv(ROOT / ".env")
DATA_DIR = ROOT / "data"

DOCUMENTOS = {
    "cafeteria":   ("cafeteria_nebula_manual.md",      "cafeteria_preguntas.json"),
    "streamly":    ("streamly_api_docs.md",            "streamly_preguntas.json"),
    "veterinaria": ("clinica_veterinaria_huellitas.md", "veterinaria_preguntas.json"),
    "mi_documento": ("mi_documento.md",                 None),  # genera el tuyo con los prompts de alumnos/02_material_del_alumno.md
}

def cargar_documento(nombre):
    archivo, _ = DOCUMENTOS[nombre]
    return (DATA_DIR / archivo).read_text(encoding="utf-8")

def cargar_preguntas(nombre):
    _, archivo = DOCUMENTOS[nombre]
    if archivo is None:
        return []
    return json.loads((DATA_DIR / archivo).read_text(encoding="utf-8"))

print("Repositorio:", ROOT)

In [ ]:
# 👉 Elige el documento con el que vas a trabajar
DOCUMENTO = "cafeteria"   # "cafeteria" | "streamly" | "veterinaria" | "mi_documento"

texto = cargar_documento(DOCUMENTO)
print(f"Documento: {DOCUMENTO} — {len(texto):,} caracteres, {len(texto.split()):,} palabras")
print(texto[:500], "...")

In [ ]:
# Funciones de chunking (las construiste en el LAB 02)
import re

def chunk_text(text, chunk_size=500, overlap=100):
    if overlap >= chunk_size:
        raise ValueError("overlap debe ser menor que chunk_size")
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start:start + chunk_size])
        start += chunk_size - overlap
    return chunks

def chunk_by_section(text, max_chars=1200):
    """Divide por encabezados Markdown y subdivide secciones largas por párrafos."""
    resultado, seccion, buffer = [], "Inicio", []

    def vaciar():
        cuerpo = "\n".join(buffer).strip()
        if not cuerpo:
            return
        parrafos, actual = [p for p in cuerpo.split("\n\n") if p.strip()], ""
        for p in parrafos:
            if actual and len(actual) + len(p) > max_chars:
                resultado.append({"seccion": seccion, "texto": f"{seccion}\n{actual.strip()}"})
                actual = ""
            actual += p + "\n\n"
        if actual.strip():
            resultado.append({"seccion": seccion, "texto": f"{seccion}\n{actual.strip()}"})

    for linea in text.splitlines():
        m = re.match(r"^#{1,6}\s+(.*)", linea)
        if m:
            vaciar()
            seccion, buffer = m.group(1).strip(), []
        else:
            buffer.append(linea)
    vaciar()
    return resultado

In [ ]:
chunks = [c["texto"] for c in chunk_by_section(texto, max_chars=1200)]
print(f"{len(chunks)} chunks")

In [ ]:
# Preguntas que usan palabras DIFERENTES a las del documento
CONSULTAS = {
    "cafeteria":   ["¿Cada cuánto se le quita el sarro a la cafetera?", "¿Puedo pedir que me lleven el café a mi casa?"],
    "streamly":    ["¿Cuántas peticiones puedo hacer por minuto?", "¿Qué hago si la API me dice que hice demasiadas llamadas?"],
    "veterinaria": ["¿Qué hago después de que operen a mi perro?", "Mi perro se comió una barra de chocolate"],
}.get(DOCUMENTO, ["Escribe aquí una pregunta sobre tu documento"])

## 1. Búsqueda literal (por palabras)

### ✏️ Ejercicio 1 — `busqueda_literal`

Puntaje = número de palabras de la consulta (en minúsculas, de 4+ letras) que aparecen en el chunk.
Devuelve una lista de `(puntaje, indice)` ordenada de mayor a menor.

In [ ]:
def palabras(t):
    return set(re.findall(r"\w{4,}", t.lower()))

def busqueda_literal(consulta, chunks, k=3):
    # TODO
    ...

for q in CONSULTAS:
    print("🔎", q)
    for score, i in busqueda_literal(q, chunks):
        print(f"   {score} palabras | {chunks[i][:90]!r}")

In [ ]:
# ✅ Autoverificación
r = busqueda_literal("horario de soporte", ["el horario es largo", "soporte y horario", "nada"], k=2)
assert r[0] == (2, 1)
print("✅ Ejercicio 1 correcto")

## 2. TF-IDF: búsqueda literal "mejorada"

TF-IDF da más peso a palabras raras y menos a palabras comunes, pero **sigue dependiendo de que las palabras coincidan**.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(chunks)

for q in CONSULTAS:
    scores = cosine_similarity(vectorizer.transform([q]), X)[0]
    mejor = scores.argmax()
    print(f"🔎 {q}\n   score={scores[mejor]:.3f} | {chunks[mejor][:100]!r}\n")

## 3. Embeddings

Configuramos el proveedor (OpenAI u Ollama según tu `.env`). `get_embeddings` ya está hecha y cachea resultados.

In [ ]:
# Proveedor híbrido: OpenAI (API) u Ollama (local). Se configura en .env con LLM_PROVEEDOR.
# Ollama expone un endpoint compatible con la API de OpenAI, así que usamos el mismo cliente.
import numpy as np
from openai import OpenAI

PROVEEDOR = os.getenv("LLM_PROVEEDOR", "openai").lower()

if PROVEEDOR == "openai":
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("Falta OPENAI_API_KEY en .env (o cambia LLM_PROVEEDOR=ollama).")
    client = OpenAI()
    EMBEDDING_MODEL = "text-embedding-3-small"
    CHAT_MODEL = "gpt-4.1-mini"
elif PROVEEDOR == "ollama":
    client = OpenAI(base_url=os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1"), api_key="ollama")
    EMBEDDING_MODEL = os.getenv("OLLAMA_EMBEDDING_MODEL", "nomic-embed-text")
    CHAT_MODEL = os.getenv("OLLAMA_CHAT_MODEL", "llama3.2")
else:
    raise ValueError("LLM_PROVEEDOR debe ser 'openai' u 'ollama'")

# Nota: los nombres de modelos cambian con el tiempo; verifica la documentación vigente del proveedor.
print(f"Proveedor: {PROVEEDOR} | embeddings: {EMBEDDING_MODEL} | chat: {CHAT_MODEL}")

_cache_embeddings = {}

def get_embeddings(textos, batch_size=64):
    """Devuelve una matriz (n_textos, dimensiones). Cachea resultados para no pagar dos veces."""
    faltantes = [t for t in dict.fromkeys(textos) if t not in _cache_embeddings]
    for i in range(0, len(faltantes), batch_size):
        lote = faltantes[i:i + batch_size]
        resp = client.embeddings.create(model=EMBEDDING_MODEL, input=lote)
        for t, item in zip(lote, resp.data):
            _cache_embeddings[t] = item.embedding
    return np.array([_cache_embeddings[t] for t in textos], dtype=float)

In [ ]:
# Calentamiento: el ejemplo de la diapositiva
frases = [
    "¿Cómo reemplazo la batería?",
    "Procedimiento para sustituir el acumulador.",
    "Revisión del sistema de frenos.",
]
E = get_embeddings(frases)
print("Forma de la matriz:", E.shape, "→ cada texto es un vector de", E.shape[1], "números")
print("Primeros valores del primer vector:", np.round(E[0][:6], 3))

## 4. Cosine similarity

`cos(A, B) = (A · B) / (‖A‖ ‖B‖)`

### ✏️ Ejercicio 2 — `cosine`

In [ ]:
def cosine(a, b):
    # TODO (pista: np.dot y np.linalg.norm; cuida la división entre cero)
    ...

print("batería vs acumulador:", round(cosine(E[0], E[1]), 3))
print("batería vs frenos    :", round(cosine(E[0], E[2]), 3))

In [ ]:
# ✅ Autoverificación
assert abs(cosine(np.array([1, 0]), np.array([1, 0])) - 1) < 1e-9
assert abs(cosine(np.array([1, 0]), np.array([0, 1]))) < 1e-9
assert abs(cosine(np.array([1, 1]), np.array([-1, -1])) + 1) < 1e-9
assert cosine(np.array([0, 0]), np.array([1, 1])) == 0.0
print("✅ Ejercicio 2 correcto")

## 5. Indexar los chunks y buscar Top-K

### ✏️ Ejercicio 3 — `top_k`

Recibe el embedding de la consulta y la matriz de embeddings de los chunks.
Devuelve una lista de diccionarios `{"indice", "score"}` con los `k` más similares, de mayor a menor.

In [ ]:
# Pasos 1 y 2: crear y guardar los embeddings de los chunks (nuestro "vector store" más simple: una matriz)
matriz = get_embeddings(chunks)
print("Matriz de embeddings:", matriz.shape)

In [ ]:
def top_k(query_emb, matriz, k=3):
    # TODO: calcula el coseno contra cada fila, ordena y devuelve los k mejores
    ...

In [ ]:
# ✅ Autoverificación
M = np.array([[1, 0], [0, 1], [0.9, 0.1]])
r = top_k(np.array([1, 0]), M, k=2)
assert [x["indice"] for x in r] == [0, 2]
print("✅ Ejercicio 3 correcto")

In [ ]:
def buscar(consulta, k=3):
    q_emb = get_embeddings([consulta])[0]          # paso 3: la pregunta también se convierte en vector
    return top_k(q_emb, matriz, k)                  # pasos 4-6: comparar, ordenar, Top-K

for q in CONSULTAS:
    print("🔎", q)
    for r in buscar(q):
        print(f"   score={r['score']:.3f} | {chunks[r['indice']][:90]!r}")
    print()

## 6. Literal vs. semántica, lado a lado

Para cada consulta, ¿el primer resultado es el mismo? ¿Cuál es correcto?

In [ ]:
import pandas as pd
filas = []
for q in CONSULTAS:
    lit = busqueda_literal(q, chunks, 1)[0][1]
    sem = buscar(q, 1)[0]["indice"]
    filas.append({"consulta": q, "literal": chunks[lit].split("\n")[0], "semántica": chunks[sem].split("\n")[0]})
pd.DataFrame(filas)

## 7. ¿Cuánto debe valer K?

Escribe tu propia pregunta y observa los scores de los primeros 8 resultados. ¿Dónde caen los scores?
¿A partir de qué posición los resultados ya no son relevantes?

In [ ]:
MI_PREGUNTA = CONSULTAS[0]   # 👉 cámbiala
for pos, r in enumerate(buscar(MI_PREGUNTA, k=8), 1):
    print(f"{pos}. score={r['score']:.3f} | {chunks[r['indice']].splitlines()[0]}")

## 8. (Opcional) Visualiza el espacio de embeddings en 2D

Reducimos las dimensiones con PCA para poder dibujarlas. Es una aproximación: se pierde información.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

q_embs = get_embeddings(CONSULTAS)
puntos = PCA(n_components=2).fit_transform(np.vstack([matriz, q_embs]))

plt.figure(figsize=(9, 6))
plt.scatter(puntos[:len(chunks), 0], puntos[:len(chunks), 1], c="steelblue", label="chunks")
for i, c in enumerate(chunks):
    plt.annotate(c.splitlines()[0][:22], puntos[i], fontsize=7)
plt.scatter(puntos[len(chunks):, 0], puntos[len(chunks):, 1], c="orangered", marker="*", s=200, label="consultas")
plt.legend(); plt.title("Chunks y consultas en 2D (PCA)"); plt.show()

## Preguntas de cierre

1. ¿En qué consulta falló la búsqueda literal y acertó la semántica? ¿Por qué?
2. ¿Encontraste algún caso donde la semántica se equivocara?
3. ¿Qué K elegirías para tu documento? ¿Qué pasa con K = 1 y con K = 10?